# Power Spectrum Showcase

This notebook demonstrates the enhanced `PowerSpectrum` class and the new utilities for computing transfer functions and coherence.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from fwd_model_tools.power import PowerSpectrum, compute_pk, compute_transfer, compute_coherence
from fwd_model_tools.fields import DensityField

%matplotlib inline

## 1. Creating Synthetic Data

We create two Gaussian random fields to simulate density fields. One is a slightly modified version of the other to make the transfer function interesting.

In [ ]:
N = 64
L = 100.0
shape = (N, N, N)
box_size = (L, L, L)

key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)

# Field 1: Gaussian noise
field1_arr = jax.random.normal(k1, shape)

# Field 2: Correlated with Field 1 + some noise
# Transfer function T(k) ~ 0.8
field2_arr = 0.8 * field1_arr + 0.5 * jax.random.normal(k2, shape)

field1 = DensityField(array=field1_arr, mesh_size=shape, box_size=box_size)
field2 = DensityField(array=field2_arr, mesh_size=shape, box_size=box_size)

## 2. Computing Power Spectra

We compute the auto-power spectra of both fields.

In [ ]:
pk1 = field1.compute_power_spectrum(label="Field 1")
pk2 = field2.compute_power_spectrum(label="Field 2")

print("Pk1:", pk1)
print("Pk2:", pk2)

## 3. Plotting Individual Spectra

We can plot them individually using the new `plot` method.

In [ ]:
fig, ax, _ = pk1.plot(color='blue')
pk2.plot(ax=ax, color='red')
plt.show()

## 4. Comparison with Ratio

Use the `compare` method to show the ratio between the two spectra.

In [ ]:
pk1.compare([pk2], show_ratio=True, shaded_regions=[0.1, 0.2])
plt.show()

## 5. Transfer Function and Coherence

Compute and plot the transfer function and coherence.

In [ ]:
transfer = field1.compute_transfer(field2)
coherence = field1.compute_coherence(field2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

transfer.plot(ax=ax1, logy=False, label="Transfer Function")
ax1.axhline(0.8, color='k', linestyle='--', label="Expected (0.8)")
ax1.legend()
ax1.set_title("Transfer Function T(k) = P_12 / P_11")

coherence.plot(ax=ax2, logy=False, label="Coherence")
ax2.set_ylim(0, 1.1)
ax2.set_title("Coherence r(k)")

plt.tight_layout()
plt.show()

## 6. Batch Processing and Mean/Std Plotting

Demonstrate handling multiple spectra at once.

In [ ]:
# Create a batch of 10 noisy spectra based on pk1
n_realizations = 10
noise_scale = 0.2

# We'll just simulate the Pk values directly for this demo
pk_batch_values = []
for i in range(n_realizations):
    noise = 1.0 + noise_scale * np.random.randn(pk1.pk.size)
    pk_batch_values.append(pk1.pk * noise)

pk_batch_values = jnp.array(pk_batch_values)

pk_batch = PowerSpectrum(k=pk1.k, pk=pk_batch_values, name="pk_batch")

# Plot all individual lines
pk_batch.plot(mode="individual", alpha=0.5)
plt.title("Individual Realizations")
plt.show()

# Plot mean and std
pk_batch.plot(mode="mean_std", color="green", alpha=0.3)
plt.title("Mean ± Std")
plt.show()